In [7]:
%%writefile my_streamlit_app.py
import streamlit as st
import cv2
import numpy as np

st.title("OpenCV + Streamlit in Jupyter")
st.write("This app was created inside a Notebook!")

# Example: Display an image or use your CV2 code here
uploaded_file = st.file_uploader("Choose an image...", type="jpg")
if uploaded_file is not None:
    st.image(uploaded_file, caption='Uploaded Image.', use_column_width=True)

Writing my_streamlit_app.py


In [1]:
%%writefile my_streamlit_app.py
import streamlit as st
import pandas as pd
import datetime
import cv2
import time
import os
import numpy as np

st.set_page_config(page_title="AI Study Assistant", page_icon="📚")
st.title("AI Study Assistant 📚")

# -------- Load or Create Task File --------
# Using a CSV to keep data even if the app restarts
if not os.path.exists("tasks.csv"):
    df = pd.DataFrame(columns=["Task", "Deadline"])
    df.to_csv("tasks.csv", index=False)
else:
    df = pd.read_csv("tasks.csv")

# -------- Add Task --------
st.header("Add Task")
with st.form("task_form"):
    task = st.text_input("Enter Task")
    deadline = st.date_input("Select Deadline")
    submit = st.form_submit_button("Add Task")

if submit and task:
    new_task = pd.DataFrame([[task, deadline]], columns=["Task", "Deadline"])
    df = pd.concat([df, new_task], ignore_index=True)
    df.to_csv("tasks.csv", index=False)
    st.success(f"Task '{task}' Added!")
    st.rerun() # Refresh to show new task in the table below

# -------- Show Tasks --------
st.header("Your Tasks")
st.dataframe(df, use_container_width=True)

# -------- Simple Scheduler --------
def suggest_time():
    hour = datetime.datetime.now().hour
    if hour < 12:
        return "Morning (9 AM - 12 PM)"
    elif hour < 18:
        return "Afternoon (2 PM - 5 PM)"
    else:
        return "Evening (7 PM - 10 PM)"

if st.button("Generate Smart Schedule"):
    st.info(f"Recommended Study Time: {suggest_time()}")

# -------- Focus Guardian 👀 --------
st.header("Focus Guardian 👀")
st.write("Click below to check your focus level for the next 5 seconds.")

def focus_guardian():
    cap = cv2.VideoCapture(0)
    # Load the pre-trained face detection model
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

    start_time = time.time()
    focus_count = 0
    total_frames = 0
    
    # Placeholder for the webcam feed in the UI
    frame_placeholder = st.empty()

    while time.time() - start_time < 5:
        ret, frame = cap.read()
        if not ret:
            st.error("Cannot access webcam.")
            break

        # Convert to grayscale for detection
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, 1.3, 5)

        # Draw rectangles around detected faces (Optional visual feedback)
        for (x, y, w, h) in faces:
            cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)

        # Update the web app with the live frame
        frame_placeholder.image(frame, channels="BGR")

        total_frames += 1
        if len(faces) > 0:
            focus_count += 1
        
        time.sleep(0.05) # Small delay to save CPU

    cap.release()
    frame_placeholder.empty() # Remove webcam feed after check

    if total_frames == 0:
        return 0
    
    return (focus_count / total_frames) * 100

if st.button("Start Focus Check (5 sec)"):
    with st.spinner("Monitoring focus..."):
        score = focus_guardian()
        
    if score > 60:
        st.success(f"Great focus! 🔥 Your focus score: {int(score)}%")
    elif score == 0:
        st.error("No face detected. Make sure you are in front of the camera!")
    else:
        st.warning(f"Distracted ❌ Score: {int(score)}%")
        st.info("Take a short break! ☕")

Overwriting my_streamlit_app.py


In [ ]:
!streamlit run my_streamlit_app.py

In [6]:
import cv2
print(cv2.__version__)

4.13.0


In [ ]:
import sys
print(sys.executable)

In [ ]:
python -m pip install opencv-python